In [ ]:
from abc import ABC, abstractmethod
import uuid
from wordle_game import WordleGame, CORRECT, WRONG_POS, NOT_IN_WORD, MAX_GUESSES
import random
import re
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

In [ ]:
class WordleSolver(ABC):
    def __init__(self, word_list: set):
        self.bot_id = str(uuid.uuid4())
        self.full_word_list = word_list
        self.candidates = set(word_list) 
        self.history = []

    @abstractmethod
    def make_guess(self) -> str:
        """Logic to pick the next word from self.candidates."""
        pass

    @abstractmethod
    def handle_feedback(self, guess: str, feedback: str):
        """Logic to prune self.candidates based on the +/-* string."""
        pass

    def reset(self):
        """Crucial for experiments: clear history and reset candidates."""
        self.candidates = set(self.full_word_list)
        self.history = []

In [ ]:
class HumanWordleSolver(WordleSolver):
    def __init__(self, word_list: set):
        super().__init__(word_list)
        # We can give the human a friendly name
        self.bot_id = "Human_Player"

    def make_guess(self) -> str:
        """Prompts the user for input via the notebook/console."""
        guess = input("Enter your 5-letter guess: ").strip().lower()
        
        # Basic validation before sending it to the game engine
        while len(guess) != 5:
            print("Invalid length! Word must be 5 letters.")
            guess = input("Enter your 5-letter guess: ").strip().lower()
            
        return guess

    def handle_feedback(self, guess: str, feedback: str):
        """
        For a human, we just print the feedback so you can 
        decide your next move.
        """
        # We can use our emoji logic here to make it look nice
        emoji_map = {"+": "🟩", "*": "🟨", "-": "⬛"}
        visual_feedback = "".join(emoji_map.get(c, c) for c in feedback)
        
        print(f"\nGuess: {guess.upper()}")
        print(f"Result: {visual_feedback} ({feedback})")

In [ ]:
class RegexWordleSolver(WordleSolver):
    def __init__(self, word_list: set):
        super().__init__(word_list)

    def make_guess(self) -> str:
        """
        Strategy: Randomly sample from the remaining candidates.
        In a future iteration, we could pick words with high vowel density.
        """
        if not self.candidates:
            return "??????"  # Should not happen if logic is sound
        return random.choice(list(self.candidates))

    def handle_feedback(self, guess: str, feedback: str):
        """
        Orchestrates the pruning process after receiving feedback.
        """
        self.history.append((guess, feedback))
        self._prune_candidates(guess, feedback)

    def _prune_candidates(self, guess, feedback):
        """
        The Core Logic:
        1. Build a positional regex (e.g., ^[^b]o[^r]..$)
        2. Identify letters that MUST be in the word (Yellows/Greens).
        3. Identify letters that are truly GONE.
        """
        positional_slots = ["."] * 5
        required_letters = set()
        strictly_forbidden = set()
        
        # We need to know which letters are 'active' (Green or Yellow) 
        # to handle the duplicate letter edge case correctly.
        active_letters = {guess[i] for i, f in enumerate(feedback) if f in (CORRECT, WRONG_POS)}

        for i, (char, status) in enumerate(zip(guess, feedback)):
            if status == CORRECT:
                positional_slots[i] = char
                required_letters.add(char)
            
            elif status == WRONG_POS:
                # Must be in word, but NOT at this specific index
                positional_slots[i] = f"[^{char}]"
                required_letters.add(char)
            
            elif status == NOT_IN_WORD:
                # Only globally forbid if it's not green/yellow elsewhere in this guess
                if char not in active_letters:
                    strictly_forbidden.add(char)
                else:
                    # If it IS active elsewhere, it's just forbidden at THIS index
                    positional_slots[i] = f"[^{char}]"

        # Construct the final positional regex
        # We replace any remaining '.' with a character class excluding forbidden letters
        forbidden_pattern = f"[^{''.join(strictly_forbidden)}]" if strictly_forbidden else "."
        regex_parts = [slot if slot != "." else forbidden_pattern for slot in positional_slots]
        pattern = re.compile(f"^{''.join(regex_parts)}$")

        # The Funnel: Apply the positional regex and the inclusion requirements
        new_candidates = set()
        for word in self.candidates:
            if pattern.match(word):
                # Ensure all required (Green/Yellow) letters are present at least once
                if required_letters.issubset(set(word)):
                    new_candidates.add(word)
        
        self.candidates = new_candidates

    def reset(self):
        self.candidates = set(self.full_word_list)
        self.history = []

In [ ]:
class FrequencyWordleSolver(RegexWordleSolver):
    def __init__(self, word_list: set):
        super().__init__(word_list)

    def make_guess(self) -> str:
        """
        Greedy Heuristic: Picks the word that uses the most 'popular' 
        letters across the current candidate pool.
        """
        if not self.candidates:
            return "??????"
        
        if len(self.candidates) == 1:
            return list(self.candidates)[0]

        # 1. Calculate how often each letter appears in the remaining candidates
        letter_counts = Counter("".join(self.candidates))
        
        # 2. Score each word based on unique letter frequencies
        # We use a set of chars because repeating 'E' doesn't give new info
        best_word = None
        highest_score = -1
        
        for word in self.candidates:
            # Formula: Score = sum of frequencies of unique letters in word
            score = sum(letter_counts[char] for char in set(word))
            
            if score > highest_score:
                highest_score = score
                best_word = word
                
        return best_word

In [ ]:
import torch
from collections import Counter

class VectorizedGPUSolver(RegexWordleSolver):
    def __init__(self, word_list: set):
        super().__init__(word_list)
        self.device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
        
        # Pre-process the dictionary into GPU memory
        self.all_words_list = sorted(list(word_list))
        self.word_to_idx = {w: i for i, w in enumerate(self.all_words_list)}
        
        # Word Tensor: (N, 5)
        self.tensor_dict = torch.tensor([[ord(c)-97 for c in w] for w in self.all_words_list], 
                                        dtype=torch.uint8, device=self.device)
        
        # Frequency Tensor (Bag of Words): (N, 26)
        self.freq_dict = torch.zeros((len(self.all_words_list), 26), device=self.device)
        for i in range(5):
            self.freq_dict.scatter_add_(1, self.tensor_dict[:, i:i+1].long(), 
                                      torch.ones((len(self.all_words_list), 1), device=self.device))

    def make_guess(self) -> str:
        if len(self.candidates) <= 2:
            return list(self.candidates)[0]

        # 1. Get Indices for current state
        cand_idx = torch.tensor([self.word_to_idx[w] for w in self.candidates], device=self.device)
        
        # Limit search pool to top 200 to stay within M2 memory limits for the 3D tensor
        # Even at 200, we are evaluating 200 * 14,000 = 2.8 million combinations in one shot
        pool_idx = cand_idx[:200] 
        
        # 2. Extract Subsets
        # Guesses: (P, 5) -> (P, 1, 5)
        # Candidates: (C, 5) -> (1, C, 5)
        G = self.tensor_dict[pool_idx].unsqueeze(1)
        C = self.tensor_dict[cand_idx].unsqueeze(0)
        
        # 3. Vectorized Feedback Logic
        # Calculate Greens (Positional matches)
        green_mask = (G == C) # (P, C, 5)
        green_scores = green_mask.sum(dim=2) # (P, C)
        
        # Calculate Total Letter Matches (Intersection)
        # Guess Freq: (P, 26) -> (P, 1, 26)
        # Cand Freq: (C, 26) -> (1, C, 26)
        g_freq = self.freq_dict[pool_idx].unsqueeze(1)
        c_freq = self.freq_dict[cand_idx].unsqueeze(0)
        total_matches = torch.min(g_freq, c_freq).sum(dim=2) # (P, C)
        
        # Yellows = Total matches - Greens
        yellow_scores = total_matches - green_scores
        
        # 4. Create a unique Feedback ID for every (Guess, Candidate) pair
        # We use a simple base-6 encoding: (Greens * 6) + Yellows
        # Since max green is 5 and max yellow is 5, this creates unique bucket IDs
        feedback_ids = (green_scores * 6) + yellow_scores # Shape: (P, C)

        # 5. Find the "Min-Max" word
        # For each guess (row), we want to find the size of the most common feedback ID
        best_word_idx = -1
        min_max_val = float('inf')
        
        # We iterate over the Pool (P) but the inner loop (C) is fully vectorized
        # This is the best balance for M2 memory vs speed
        for i in range(len(pool_idx)):
            # Bincount is extremely fast on the GPU for counting occurrences
            counts = torch.bincount(feedback_ids[i].long())
            max_bucket = counts.max().item()
            
            if max_bucket < min_max_val:
                min_max_val = max_bucket
                best_word_idx = pool_idx[i].item()

        return self.all_words_list[best_word_idx]

In [ ]:
import torch
if torch.backends.mps.is_available():
    print("🚀 Metal GPU is ready! Your M2 is about to fly.")
else:
    print("⚠️ MPS not found. Check your macOS version (requires 12.3+).")

In [ ]:
# %% [Full Tournament Suite: Wall Time vs CPU Time]
import time
import random
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from wordle_game import WordleGame, CORRECT, WRONG_POS, NOT_IN_WORD, MAX_GUESSES

# --- TOURNAMENT CORE FUNCTIONS ---

def run_single_trial(solver_class, secret_word, word_list):
    """
    Worker function: Runs one solver against one secret word.
    Tracks both Wall-Clock time and CPU-execution time.
    """
    # Start Clocks
    start_wall = time.perf_counter()
    start_cpu = time.process_time()
    
    # Initialize Game
    game = WordleGame(secret_word=secret_word, word_list=word_list)
    solver = solver_class(word_list)
    
    # Game Loop
    while not game.status[solver.bot_id]["over"]:
        guess = solver.make_guess()
        res = game.play(solver.bot_id, guess)
        if "error" not in res:
            solver.handle_feedback(guess, res["feedback"])
            
    # End Clocks
    end_wall = time.perf_counter()
    end_cpu = time.process_time()
    
    final_state = game.status[solver.bot_id]
    num_guesses = len(game.sessions[solver.bot_id])
    score = num_guesses if final_state["won"] else MAX_GUESSES + 1
    
    return {
        "solver": solver.__class__.__name__,
        "won": final_state["won"],
        "guesses": score,
        "wall_time_ms": (end_wall - start_wall) * 1000,
        "cpu_time_ms": (end_cpu - start_cpu) * 1000
    }

def run_tournament(solver_classes, n_trials=100):
    """
    Orchestrates tournament and aggregates wall vs cpu time.
    """
    if not words:
        print("❌ Error: No word list found.")
        return None

    test_secrets = [random.choice(tuple(words)) for _ in range(n_trials)]
    all_results = []
    
    print(f"🚀 Starting Tournament: {len(solver_classes)} solvers, {n_trials} trials each.")
    
    with ThreadPoolExecutor() as executor:
        futures = []
        for s_class in solver_classes:
            for secret in test_secrets:
                futures.append(executor.submit(run_single_trial, s_class, secret, words))
        
        for future in futures:
            all_results.append(future.result())

    df = pd.DataFrame(all_results)
    
    # Statistical Aggregation
    stats = df.groupby('solver').agg({
        'guesses': ['mean', 'median', lambda x: (x <= MAX_GUESSES).mean() * 100],
        'wall_time_ms': 'mean',
        'cpu_time_ms': 'mean'
    })
    stats.columns = ['Mean Guesses', 'Median', 'Win Rate %', 'Wall Time (ms)', 'CPU Time (ms)']
    
    print("\n" + "="*70)
    print("📈 TOURNAMENT STATISTICS")
    print("="*70)
    print(stats.round(2).to_string())
    print("-" * 70)
    
    plot_tournament_results(df)
    return df

def plot_tournament_results(df):
    plt.figure(figsize=(14, 8))
    sns.set_theme(style="whitegrid")
    guess_order = list(range(1, MAX_GUESSES + 2))
    
    ax = sns.countplot(data=df, x='guesses', hue='solver', palette="viridis", order=guess_order)
    x_labels = [str(i) for i in range(1, MAX_GUESSES + 1)] + ['Loss']
    plt.xticks(range(len(x_labels)), x_labels)
    plt.title(f"Solver Performance Comparison", fontsize=16, fontweight='bold')
    plt.legend(title="Bot Strategy", loc='upper right')
    plt.tight_layout()
    plt.show()

# --- EXECUTION ---

words = set()
try:
    with open("words.txt", "r") as f:
        words = {line.strip().lower() for line in f if len(line.strip()) == 5}
    print(f"✅ Dictionary Loaded: {len(words)} words ready.")
except FileNotFoundError:
    print("❌ Error: words.txt not found!")

if words:
    # Adding the GPU solver we just built
    competitors = [RegexWordleSolver, FrequencyWordleSolver, VectorizedGPUSolver] 
    trials_per_bot = 500
    tournament_df = run_tournament(competitors, n_trials=trials_per_bot)